<a href="https://colab.research.google.com/github/avyue/datasci112_finalproject/blob/main/(Poverty_Rate)_HUD_Qualified_Census_Tracts_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import numpy as np
import pandas as pd
import plotly.express as px
pd.set_option("display.max_columns", 60)

In [ ]:
ACS_YEAR = 2023      # 2019-2023 5-year. Use 2022 to match HUD's 2025 QCT inputs; try 2024 if released.
CENSUS_KEY = "c1c3aa7476f3f166756b7a5c434d84b3b1420ec9"
VARS = ["NAME", "B17001_001E", "B17001_002E", "B19013_001E", "B19001_001E"]

url = f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5"
params = {"get": ",".join(VARS), "for": "tract:*", "in": "state:06 county:037", "key": CENSUS_KEY}
r = requests.get(url, params=params, timeout=120)
r.raise_for_status()
rows = r.json()
acs = pd.DataFrame(rows[1:], columns=rows[0])
print("shape:", acs.shape)
acs.head(3)

shape: (2498, 8)


,NAME,B17001_001E,B17001_002E,B19013_001E,B19001_001E,state,county,tract
0,Census Tract 1011.10; Los Angeles County; Cali...,4068,476,84091,1558,06,037,101110
1,Census Tract 1011.22; Los Angeles County; Cali...,4166,266,99583,1407,06,037,101122
2,Census Tract 1012.20; Los Angeles County; Cali...,3434,610,69676,1357,06,037,101220


In [ ]:
num = ["B17001_001E", "B17001_002E", "B19013_001E", "B19001_001E"]
for c in num:
    acs[c] = pd.to_numeric(acs[c], errors="coerce")

# Census codes missing median income as a large negative sentinel -> NaN
acs["B19013_001E"] = acs["B19013_001E"].where(acs["B19013_001E"] > 0, np.nan)

acs = acs.rename(columns={"B17001_001E": "pov_universe", "B17001_002E": "pov_below",
                          "B19013_001E": "median_income", "B19001_001E": "households"})
acs["GEOID"]        = acs["state"] + acs["county"] + acs["tract"]   # 11-digit
acs["tract_ce6"]    = acs["tract"].str.zfill(6)                     # matches your LAHSA key
acs["poverty_rate"] = (acs["pov_below"] / acs["pov_universe"]).where(acs["pov_universe"] > 0)

acs[["GEOID", "tract_ce6", "pov_universe", "pov_below", "poverty_rate",
     "median_income", "households"]].head()

,GEOID,tract_ce6,pov_universe,pov_below,poverty_rate,median_income,households
0,06037101110,101110,4068,476,0.117011,84091.0,1558
1,06037101122,101122,4166,266,0.063850,99583.0,1407
2,06037101220,101220,3434,610,0.177635,69676.0,1357
3,06037101221,101221,3881,555,0.143004,53798.0,1483
4,06037101222,101222,2564,356,0.138846,45662.0,948


In [ ]:
def fetch_arcgis_attrs(item_id, fields, layer=0, page=2000):
    info = requests.get(f"https://www.arcgis.com/sharing/rest/content/items/{item_id}",
                        params={"f": "json"}, timeout=60).json()
    qurl = f"{info['url']}/{layer}/query"
    rows, offset = [], 0
    while True:
        p = {"where": "1=1", "outFields": fields, "returnGeometry": "false",
             "f": "json", "resultOffset": offset, "resultRecordCount": page}
        js = requests.get(qurl, params=p, timeout=120).json()
        batch = [f["attributes"] for f in js.get("features", [])]
        rows.extend(batch)
        if len(batch) < page:
            break
        offset += page
    return pd.DataFrame(rows)

xwalk = fetch_arcgis_attrs("35beb4d4ad324cac954c5c840a724285", "GEOID,spa,sup_dist")
xwalk["GEOID"]   = xwalk["GEOID"].astype(str).str.zfill(11)
xwalk["spa_num"] = xwalk["spa"].astype(str).str.extract(r"(\d+)")[0]
xwalk = xwalk[["GEOID", "spa", "spa_num", "sup_dist"]]
print("crosswalk rows:", len(xwalk))
xwalk.head(3)

crosswalk rows: 2495


,GEOID,spa,spa_num,sup_dist
0,06037294610,SPA 8 - South Bay,8,District 4
1,06037273700,SPA 5 - West,5,District 3
2,06037207501,SPA 4 - Metro,4,District 1


In [ ]:
df = acs.merge(xwalk, on="GEOID", how="left")
print("tracts with an SPA matched:", df["spa_num"].notna().sum(), "/", len(df))

df["high_poverty_25"] = df["poverty_rate"] >= 0.25   # mirrors one of the two QCT criteria
df[["GEOID", "spa_num", "poverty_rate", "median_income", "high_poverty_25"]].head()


tracts with an SPA matched: 2495 / 2498


,GEOID,spa_num,poverty_rate,median_income,high_poverty_25
0,06037101110,2,0.117011,84091.0,False
1,06037101122,2,0.063850,99583.0,False
2,06037101220,2,0.177635,69676.0,False
3,06037101221,2,0.143004,53798.0,False
4,06037101222,2,0.138846,45662.0,False


In [ ]:
tract_out = df[["GEOID", "tract_ce6", "spa", "spa_num", "sup_dist",
                "pov_universe", "pov_below", "poverty_rate",
                "median_income", "households", "high_poverty_25"]].copy()
tract_out.to_csv("poverty_signals_by_tract.csv", index=False)
print("Saved poverty_signals_by_tract.csv", tract_out.shape)
tract_out.head()

Saved poverty_signals_by_tract.csv (2498, 11)


,GEOID,tract_ce6,spa,spa_num,sup_dist,pov_universe,pov_below,poverty_rate,median_income,households,high_poverty_25
0,06037101110,101110,SPA 2 - San Fernando,2,District 5,4068,476,0.117011,84091.0,1558,False
1,06037101122,101122,SPA 2 - San Fernando,2,District 5,4166,266,0.063850,99583.0,1407,False
2,06037101220,101220,SPA 2 - San Fernando,2,District 5,3434,610,0.177635,69676.0,1357,False
3,06037101221,101221,SPA 2 - San Fernando,2,District 5,3881,555,0.143004,53798.0,1483,False
4,06037101222,101222,SPA 2 - San Fernando,2,District 5,2564,356,0.138846,45662.0,948,False


In [ ]:
g = df.dropna(subset=["spa_num"]).groupby("spa_num")
spa_out = g.agg(n_tracts=("GEOID", "size"),
                pov_universe=("pov_universe", "sum"),
                pov_below=("pov_below", "sum"),
                pct_tracts_high_poverty=("high_poverty_25", "mean")).reset_index()
spa_out["poverty_rate"] = spa_out["pov_below"] / spa_out["pov_universe"]

# household-weighted median income per SPA (approximation)
wi = (df.dropna(subset=["spa_num", "median_income", "households"])
        .groupby("spa_num")
        .apply(lambda x: np.average(x["median_income"], weights=x["households"]))
        .rename("median_income_wt").reset_index())
spa_out = spa_out.merge(wi, on="spa_num", how="left")

spa_out.to_csv("poverty_signals_by_spa.csv", index=False)
print("Saved poverty_signals_by_spa.csv", spa_out.shape)
spa_out

Saved poverty_signals_by_spa.csv (8, 7)


/tmp/ipykernel_33181/3153920211.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: np.average(x["median_income"], weights=x["households"]))


,spa_num,n_tracts,pov_universe,pov_below,pct_tracts_high_poverty,poverty_rate,median_income_wt
0,1,93,400983,58426,0.118280,0.145707,81920.978852
1,2,558,2158094,262676,0.086022,0.121717,102982.157496
2,3,392,1691945,188626,0.028061,0.111485,99589.048277
3,4,352,1089775,204940,0.247159,0.188057,79440.187590
4,5,185,619121,61640,0.032432,0.099561,129000.831548
5,6,245,983912,212026,0.363265,0.215493,63248.828145
6,7,290,1241991,149499,0.058621,0.120370,88335.755745
7,8,380,1507606,184643,0.081579,0.122474,101208.980300


In [ ]:
from google.colab import files
files.download("poverty_signals_by_tract.csv")
files.download("poverty_signals_by_spa.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import plotly.express as px
px.bar(spa_out.sort_values("poverty_rate"), x="poverty_rate", y="spa_num",
       orientation="h", title=f"Poverty rate by SPA (ACS {ACS_YEAR} 5-year)").show()